# Workshop Part 5: Deployment Theory

## Learning Objectives
- Understand different ML deployment strategies
- Learn about model serving (batch vs. real-time)
- Explore MLOps concepts
- See real-world deployment examples

**Note**: This section is **theoretical only**. We won't deploy the model, but we'll discuss how it would be done in production.

## Why Deployment Matters

A model is only valuable when it's **used**:
- Training a model in a notebook = 20% of the work
- Deploying and maintaining it in production = 80% of the work

**Goal**: Get predictions from our sentiment model into the hands of store managers and POS systems.

In [ ]:
# Workshop Progress Tracker
notebooks = ["01 Extract", "02 Prepare", "03 Storage", "04 ML", "05 Deploy"]
current = 4  # This is notebook 05

print("="*70)
print("📊 WORKSHOP PROGRESS")
print("="*70)
for i, nb in enumerate(notebooks):
    if i < current:
        print(f"✅ {nb}")
    elif i == current:
        print(f"👉 {nb} ← YOU ARE HERE")
    else:
        print(f"⬜ {nb}")
print("="*70)

## 1. Model Deployment Options

### A. Batch Predictions (Scheduled Jobs)

**How it works:**
- Run predictions on a schedule (e.g., daily at 6 AM)
- Process all new reviews in bulk
- Write results to a database or file
- Business users access results via dashboards

**Architecture:**
```
┌──────────────┐     Schedule      ┌──────────────────┐
│              │  (e.g., Daily)    │                  │
│  Azure Blob  │ ───────────────> │  Azure Databricks│
│  Storage     │                   │  (Batch Job)     │
│ (Raw Reviews)│                   │                  │
└──────────────┘                   └──────────────────┘
                                            │
                                            │ Write predictions
                                            ↓
                                   ┌──────────────────┐
                                   │  Azure SQL DB    │
                                   │  (Predictions)   │
                                   └──────────────────┘
                                            │
                                            │ Query
                                            ↓
                                   ┌──────────────────┐
                                   │  PowerBI         │
                                   │  (Dashboard)     │
                                   └──────────────────┘
```

**Pros:**
- Simple to implement and debug
- Cost-effective (only runs when needed)
- Can process large volumes efficiently
- Easy to retry if something fails

**Cons:**
- Not real-time (predictions are delayed)
- Can't respond to immediate events

**Best for:**
- Daily/weekly reports
- Discount recommendations (prepared overnight)
- Product quality monitoring

**Example: AH Dynamic Markdown Use Case**
- Every morning at 6 AM, batch job runs
- Analyzes yesterday's reviews for all products
- Calculates sentiment-based discount recommendations
- Store managers see results in PowerBI by 8 AM

In [ ]:
# Example: Batch Prediction Script (Pseudocode)
"""
# batch_predict.py

import pandas as pd
import joblib
from datetime import datetime

# Load model and vectorizer
model = joblib.load('sentiment_model.pkl')
vectorizer = joblib.load('tfidf_vectorizer.pkl')

# Load new reviews from yesterday
yesterday = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
df = pd.read_csv(f'reviews_{yesterday}.csv')

# Make predictions
X = vectorizer.transform(df['review_text_clean'])
df['sentiment_pred'] = model.predict(X)
df['sentiment_proba'] = model.predict_proba(X).max(axis=1)

# Calculate discounts
df['recommended_discount'] = df.apply(calculate_discount, axis=1)

# Write to database
df.to_sql('sentiment_predictions', connection, if_exists='append')

print(f"Processed {len(df)} reviews for {yesterday}")
"""

print("Batch prediction script (example above) would:")
print("1. Run on a schedule (e.g., Azure Data Factory trigger)")
print("2. Load model from storage")
print("3. Process all new reviews")
print("4. Write predictions to database")
print("5. Send completion notification")

### B. Real-Time API (REST Endpoint)

**How it works:**
- Model hosted as a web service
- Applications send HTTP requests with review text
- API returns sentiment prediction immediately
- Useful for interactive applications

**Architecture:**
```
┌──────────────┐                    ┌──────────────────┐
│              │   HTTP POST        │                  │
│  POS System  │ ────────────────> │  Flask/FastAPI   │
│  or Web App  │   {review_text}    │  (API Server)    │
│              │ <──────────────── │                  │
└──────────────┘   {sentiment,     └──────────────────┘
                    discount}                │
                                             │ Load model
                                             ↓
                                    ┌──────────────────┐
                                    │  ML Model (.pkl) │
                                    │  In Memory       │
                                    └──────────────────┘
```

**Pros:**
- Real-time predictions (milliseconds)
- Interactive user experience
- Can integrate with any application

**Cons:**
- More complex to build and maintain
- Higher cost (server always running)
- Scaling challenges with high traffic
- Need to handle failures and retries

**Best for:**
- Interactive applications
- Real-time decision making
- User-facing features

In [ ]:
# Example: FastAPI REST API (Pseudocode)
"""
# api.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import re

# Load model at startup
app = FastAPI()
model = joblib.load('models/sentiment_model.pkl')
vectorizer = joblib.load('models/tfidf_vectorizer.pkl')

class ReviewRequest(BaseModel):
    product_name: str
    review_text: str

class DiscountResponse(BaseModel):
    product_name: str
    sentiment: str
    confidence: float
    recommended_discount: int

@app.post("/predict-discount", response_model=DiscountResponse)
def predict_discount(request: ReviewRequest):
    try:
        # Clean text
        text_clean = request.review_text.lower()
        text_clean = re.sub(r'[^a-z\\s.,!?]', '', text_clean)
        
        # Predict
        X = vectorizer.transform([text_clean])
        sentiment = model.predict(X)[0]
        proba = model.predict_proba(X)[0].max()
        
        # Calculate discount
        discount = calculate_discount(sentiment, proba)
        
        return DiscountResponse(
            product_name=request.product_name,
            sentiment=sentiment,
            confidence=proba,
            recommended_discount=discount
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Run with: uvicorn api:app --host 0.0.0.0 --port 8000
"""

print("Real-time API example:")
print("\nRequest:")
print("  POST http://api.example.com/predict-discount")
print("  {")
print('    "product_name": "Spinach",') 
print('    "review_text": "This is the worst spinach ever!"')
print("  }")
print("\nResponse:")
print("  {")
print('    "product_name": "Spinach",')  
print('    "sentiment": "negative",')  
print('    "confidence": 0.94,')
print('    "recommended_discount": 90')
print("  }")

### C. Serverless Functions (Azure Functions, AWS Lambda)

**How it works:**
- Deploy model as a serverless function
- Function only runs when triggered (event-driven)
- Automatically scales with demand
- Pay only for execution time

**Pros:**
- Cost-effective (pay per execution)
- Auto-scaling
- No server management

**Cons:**
- Cold start latency (first request slower)
- Execution time limits (typically 5-15 minutes)
- Model size limits

**Best for:**
- Event-driven predictions
- Low to medium traffic
- Integration with cloud services

### D. MLOps Platforms (Azure ML, Databricks, SageMaker)

**How it works:**
- Full ML lifecycle management
- Model training, versioning, deployment in one platform
- Built-in monitoring and retraining
- Enterprise-grade reliability

**Features:**
- Model registry (version control for models)
- Automated deployment pipelines
- A/B testing capabilities
- Performance monitoring
- Automated retraining

**Pros:**
- Production-ready infrastructure
- Built-in best practices
- Monitoring and governance

**Cons:**
- Higher cost
- Steeper learning curve
- Vendor lock-in

**Best for:**
- Enterprise ML deployments
- Multiple models in production
- Regulated industries (banking, healthcare)

## 2. Model Serialization

**What is model serialization?**
- Saving trained model to a file
- Can be loaded later for predictions
- Essential for deployment

**Common formats:**
- **Pickle/Joblib** (Python-specific): `.pkl` files
- **ONNX** (Language-agnostic): `.onnx` files
- **TensorFlow SavedModel**: For deep learning models
- **PMML** (Predictive Model Markup Language): XML-based

**We used Joblib in notebook 04:**

In [ ]:
import joblib
import os

# Check saved models
models_path = '../models/'
if os.path.exists(models_path):
    print("Saved models:")
    for file in os.listdir(models_path):
        if file.endswith('.pkl'):
            file_path = os.path.join(models_path, file)
            file_size = os.path.getsize(file_path) / 1024  # KB
            print(f"  - {file} ({file_size:.2f} KB)")
else:
    print("No models saved yet. Run notebook 04 first!")

print("\nTo load and use the model:")
print("  model = joblib.load('models/sentiment_model.pkl')")
print("  vectorizer = joblib.load('models/tfidf_vectorizer.pkl')")
print("  # Make predictions...")

## 3. Model Monitoring and Retraining

### Why Monitor Models?

Models can **degrade over time** due to:
- **Data drift**: Input data distribution changes
- **Concept drift**: Relationship between input and output changes
- **Model decay**: Performance naturally degrades

**Example:**
- Our sentiment model trained on 2024 reviews
- In 2025, customers use new slang: "mid" (mediocre)
- Model doesn't recognize "mid" → makes wrong predictions
- **Solution**: Monitor performance and retrain with new data

### What to Monitor

| Metric | What it Measures | Alert Threshold |
|--------|-----------------|----------------|
| **Prediction Accuracy** | How often predictions are correct | < 70% |
| **Prediction Confidence** | Model certainty in predictions | Decrease > 10% |
| **Input Distribution** | Are inputs similar to training data? | KL divergence > 0.5 |
| **Prediction Distribution** | Are outputs similar to training? | Chi-square test p < 0.05 |
| **Response Time** | API latency | > 500ms for 95th percentile |
| **Error Rate** | Failed predictions | > 1% |

### Retraining Strategy

**When to retrain:**
1. **Scheduled retraining**: Every month/quarter (proactive)
2. **Performance-based**: When accuracy drops below threshold (reactive)
3. **Data-based**: When new data volume reaches threshold (e.g., +20%)

**Retraining pipeline:**
```
┌─────────────────┐
│ Monitor         │
│ Performance     │
└────────┬────────┘
         │ Alert: accuracy < 70%
         ↓
┌─────────────────┐
│ Collect New     │
│ Training Data   │
└────────┬────────┘
         ↓
┌─────────────────┐
│ Retrain Model   │
│ (Databricks)    │
└────────┬────────┘
         ↓
┌─────────────────┐
│ Validate Model  │
│ (Test set)      │
└────────┬────────┘
         │ If better than current
         ↓
┌─────────────────┐
│ A/B Test        │
│ (10% traffic)   │
└────────┬────────┘
         │ If performance good
         ↓
┌─────────────────┐
│ Deploy New      │
│ Model (100%)    │
└─────────────────┘
```

## 4. A/B Testing

**What is A/B testing for ML?**
- Run two models in parallel
- Split traffic between them (e.g., 90% old, 10% new)
- Compare performance metrics
- Gradually shift traffic to better model

**Example:**
```
         ┌────────────────────────┐
         │   Incoming Requests    │
         └───────────┬────────────┘
                     │
                     ↓
         ┌───────────────────────┐
         │   Load Balancer       │
         │   (90/10 split)       │
         └───────────┬───────────┘
                     │
          ┌──────────┴──────────┐
          │                     │
    90%   ↓                     ↓  10%
┌──────────────┐      ┌──────────────┐
│  Model v1.0  │      │  Model v2.0  │
│  (Current)   │      │  (New)       │
└──────────────┘      └──────────────┘
```

**Metrics to compare:**
- Accuracy on same inputs
- Business metrics (e.g., waste reduction, revenue)
- User feedback
- Response time

## 5. Real-World Example: AH Dynamic Markdown Deployment

### Current Production Architecture

```
┌────────────────────────────────────────────────────────────┐
│                   DATA INGESTION (Daily)                   │
├────────────────────────────────────────────────────────────┤
│  POS Systems → Azure Event Hub → Blob Storage (Raw)        │
└────────────────────────────────────────────────────────────┘
                              ↓
┌────────────────────────────────────────────────────────────┐
│             BATCH PROCESSING (6 AM Daily)                  │
├────────────────────────────────────────────────────────────┤
│  Azure Data Factory triggers Databricks job:               │
│  1. Load raw sales + inventory data                        │
│  2. Clean and prepare features                             │
│  3. Load production model from ML Registry                 │
│  4. Generate markdown recommendations                       │
│  5. Write to Azure SQL Database                            │
└────────────────────────────────────────────────────────────┘
                              ↓
┌────────────────────────────────────────────────────────────┐
│              SERVING LAYER (Real-time)                     │
├────────────────────────────────────────────────────────────┤
│  Azure SQL Database with indexed tables:                   │
│  - product_markdown_current                                │
│  - store_level_overrides                                   │
│  - markdown_history                                        │
└────────────────────────────────────────────────────────────┘
                              ↓
┌────────────────────────────────────────────────────────────┐
│                   CONSUMPTION                              │
├────────────────────────────────────────────────────────────┤
│  • PowerBI: Store manager dashboards                       │
│  • POS Systems: Query markdown prices                      │
│  • Mobile App: Field team makes decisions                  │
└────────────────────────────────────────────────────────────┘
```

### Model Lifecycle

1. **Training (Monthly)**:
   - Databricks notebook runs model training
   - Uses last 3 months of sales data
   - Validates against last week (hold-out set)
   - Registers model in Azure ML if better

2. **Deployment (Automated)**:
   - If new model accuracy > current + 2%
   - Auto-deploy to 10% of stores (A/B test)
   - Monitor for 1 week
   - If waste reduction improves, deploy to 100%

3. **Monitoring (Continuous)**:
   - Azure Application Insights tracks:
     - Prediction latency
     - Data quality issues
     - Business KPIs (waste %, revenue)
   - Alerts sent to Slack if anomalies detected

4. **Feedback Loop**:
   - Store managers can override recommendations
   - Overrides logged and analyzed
   - Used to improve next model version

## 6. Best Practices for ML Deployment

### 1. Version Everything
- **Code**: Git for scripts and notebooks
- **Data**: DVC or Azure ML Datasets
- **Models**: Model registry with version tags
- **Dependencies**: `requirements.txt` with pinned versions

### 2. Separate Environments
- **Development**: Experiment freely
- **Staging**: Test before production
- **Production**: Stable, monitored, locked down

### 3. Implement CI/CD for ML
- Automated testing (unit tests, integration tests)
- Automated model validation
- Automated deployment pipeline

### 4. Log Everything
- Input data statistics
- Model predictions
- Performance metrics
- Errors and exceptions

### 5. Plan for Failure
- Fallback to previous model version
- Graceful degradation (default predictions)
- Circuit breakers (stop using model if errors spike)

### 6. Document Everything
- Model card: How model was trained, limitations, intended use
- API documentation: How to use the deployed model
- Runbooks: How to troubleshoot common issues

## 7. Our Workshop Model: Deployment Options

If we were to deploy our sentiment model, here are the options:

### Option 1: Batch (Recommended for Workshop Scenario)
**Why**: Discount recommendations don't need to be real-time

```python
# Daily batch job (run at 6 AM)
1. Load yesterday's reviews from Blob Storage
2. Load model: joblib.load('sentiment_model.pkl')
3. Make predictions for all reviews
4. Calculate discount recommendations
5. Write to SQL database
6. Store managers see results in PowerBI by 8 AM
```

**Cost**: Low (runs once per day)
**Complexity**: Low
**Latency**: Hours (acceptable for this use case)

### Option 2: Real-Time API (If Needed)
**Why**: If POS systems need immediate sentiment analysis

```python
# FastAPI server running on Azure App Service
POST /predict-discount
{
  "product": "Spinach",
  "review": "worst spinach ever"
}
→ {"discount": 90, "sentiment": "negative"}
```

**Cost**: Medium (server always running)
**Complexity**: Medium
**Latency**: Milliseconds

### Option 3: Serverless (Azure Functions)
**Why**: Balance between cost and responsiveness

```python
# Azure Function triggered by HTTP or Event Grid
- Scales automatically with demand
- Pay only for executions
- Good for moderate traffic
```

**Cost**: Low to Medium (pay per use)
**Complexity**: Medium
**Latency**: ~100-500ms (cold start)

## Summary: Deployment Theory

### Key Takeaways

1. **Deployment Options**:
   - Batch: Simple, cost-effective, delayed
   - Real-time API: Fast, expensive, complex
   - Serverless: Balanced, scales automatically
   - MLOps platforms: Enterprise-grade, comprehensive

2. **Model Serialization**: Save models with pickle/joblib for reuse

3. **Monitoring**: Track accuracy, confidence, latency, errors

4. **Retraining**: Schedule regular updates, monitor for drift

5. **A/B Testing**: Compare models in production safely

6. **Best Practices**:
   - Version everything
   - Separate environments
   - Automated testing and deployment
   - Comprehensive logging
   - Plan for failures

### Recommendation for Our Use Case

For AH product sentiment and discount recommendations:
- **Start with**: Batch processing (Databricks daily job)
- **Scale to**: Real-time API if business needs change
- **Monitor**: Accuracy, business KPIs (waste reduction)
- **Retrain**: Monthly with new review data

### Real-World Impact

Proper deployment means:
- Store managers have actionable insights every morning
- Products with negative sentiment get timely discounts
- Reduced food waste
- Improved customer satisfaction
- Data-driven decision making at scale

## Workshop Complete!

Congratulations! You've learned the full data lifecycle:

1. **ETL (Extract)**: Load and explore data
2. **Preparation**: Clean, transform, and prepare data
3. **Storage**: Understand where and how to store data
4. **Analysis (ML)**: Train models and make predictions
5. **Deployment**: Put models into production

### Skills You've Gained

- Data extraction and quality assessment
- Data cleaning and preprocessing
- Text analytics with TF-IDF
- Machine learning with scikit-learn
- Model evaluation and interpretation
- Understanding of deployment strategies
- Real-world data engineering concepts

### Next Steps

To continue learning:
1. Experiment with the notebooks
2. Try different ML models (Random Forest, SVM)
3. Add more features (product price, seasonality)
4. Build a simple Flask API for your model
5. Explore Azure ML or Databricks for deployment

### Questions?

Just come up to one of us to ask your question.

Thank you for participating in this workshop!